# LLVM Pass Sequence Finetuning Dataset Builder

This notebook generates a robust finetuning dataset from `llvm-test-suite` by searching for strong pass pipelines per source file.

## What It Produces
- File-level best pipeline labels (CSV + JSON)
- Optional top-k candidate history for audit/debug
- Finetuning JSONL files (`train.jsonl`, `val.jsonl`, `test.jsonl`) with chat-style records

## Robustness Features
- Resume from checkpoints (skip already processed files)
- Timeout-safe subprocess execution
- Legacy-C fallback flags
- Polybench include/link recipe handling
- Family-aware split to reduce data leakage

## Label Objective
By default, choose the pipeline that minimizes executable size among valid compilations (and optional runtime checks).

In [17]:
# Install dependencies (run once)
%pip install -q tqdm

Note: you may need to restart the kernel to use updated packages.


In [18]:
from pathlib import Path

# ---------------------------- Paths -----------------------------------------
PROJECT_ROOT = Path('.').resolve()
SUITE_DIR    = (PROJECT_ROOT / 'llvm-test-suite').resolve()
COMPARE_PY   = (PROJECT_ROOT / 'compare.py').resolve()

OUT_DIR      = (PROJECT_ROOT / 'll-files' / 'finetune-dataset').resolve()
WORK_DIR     = (OUT_DIR / 'work').resolve()
LABELS_CSV   = (OUT_DIR / 'labels_best_pipeline.csv').resolve()
LABELS_JSON  = (OUT_DIR / 'labels_best_pipeline.jsonl').resolve()
CANDIDATES_CSV = (OUT_DIR / 'candidate_trials.csv').resolve()

TRAIN_JSONL  = (OUT_DIR / 'train.jsonl').resolve()
VAL_JSONL    = (OUT_DIR / 'val.jsonl').resolve()
TEST_JSONL   = (OUT_DIR / 'test.jsonl').resolve()

# ------------------------- Data discovery -----------------------------------
# Set to subpath like 'SingleSource/Benchmarks' if you want a narrower run.
SOURCE_SUBDIR = ''
SOURCE_EXTS = {'.c'}
MAX_FILES = 0   # 0 = all files

# ------------------------ Search budget -------------------------------------
# Deterministic candidates are always evaluated.
SEED = 42
INCLUDE_SINGLE_PASS_CANDIDATES = True
INCLUDE_PAIR_PASS_CANDIDATES = True
# 0 means no cap (all pairs from PASS_POOL).
MAX_PAIR_PASS_CANDIDATES = 0

# Optional random exploration beyond deterministic candidates.
INCLUDE_RANDOM_CANDIDATES = False
MAX_RANDOM_CANDIDATES = 12
RANDOM_PIPELINE_MIN_LEN = 3
RANDOM_PIPELINE_MAX_LEN = 7

# ---------------------- Build and validation --------------------------------
SUBPROCESS_TIMEOUT_S = 45
RUN_EXECUTABLE = False  # True is stricter but slower and can fail on non-runnable tests

# ------------------------ Resume and persistence -----------------------------
RESUME = True
WRITE_CANDIDATE_LOG = True

# -------------------------- Split config ------------------------------------
TRAIN_RATIO = 0.8
VAL_RATIO   = 0.1
TEST_RATIO  = 0.1

assert abs((TRAIN_RATIO + VAL_RATIO + TEST_RATIO) - 1.0) < 1e-9

OUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('SUITE_DIR    =', SUITE_DIR)
print('OUT_DIR      =', OUT_DIR)

PROJECT_ROOT = /media/sasank-v/New Volume/Studies/College/VIT/Sem 6/Compiler Design/DA
SUITE_DIR    = /media/sasank-v/New Volume/Studies/College/VIT/Sem 6/Compiler Design/DA/llvm-test-suite
OUT_DIR      = /media/sasank-v/New Volume/Studies/College/VIT/Sem 6/Compiler Design/DA/ll-files/finetune-dataset


In [19]:
import ast
import csv
import hashlib
import itertools
import json
import os
import random
import shutil
import subprocess
import time

from tqdm.auto import tqdm

RNG = random.Random(SEED)

PASS_POOL = [
    'mem2reg', 'instcombine', 'simplifycfg', 'gvn', 'licm',
    'loop-unroll', 'dce', 'inline', 'sccp', 'adce',
    'tailcallelim', 'jump-threading', 'early-cse',
    'sroa', 'reassociate', 'loop-rotate', 'indvars',
]

LEGACY_C_FLAGS = [
    '-std=gnu89',
    '-Wno-error=implicit-int',
    '-Wno-error=implicit-function-declaration',
    '-Wno-error=deprecated-non-prototype',
]

def run_cmd(args, timeout_s=SUBPROCESS_TIMEOUT_S, cwd=None):
    """Safe subprocess wrapper. Returns (ok, stdout, stderr, elapsed_s, timed_out)."""
    t0 = time.perf_counter()
    try:
        p = subprocess.run(
            args,
            cwd=cwd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            timeout=timeout_s,
        )
        elapsed = time.perf_counter() - t0
        return p.returncode == 0, p.stdout, p.stderr, elapsed, False
    except subprocess.TimeoutExpired as e:
        elapsed = time.perf_counter() - t0
        return False, e.stdout or '', e.stderr or 'timeout', elapsed, True

def read_csv_rows(path: Path):
    if not path.exists() or path.stat().st_size == 0:
        return []
    with path.open('r', encoding='utf-8', newline='') as f:
        return list(csv.DictReader(f))

def find_source_root():
    base = SUITE_DIR if SOURCE_SUBDIR.strip() == '' else (SUITE_DIR / SOURCE_SUBDIR)
    if not base.exists():
        raise FileNotFoundError(f'Source root not found: {base}')
    return base.resolve()

def discover_sources(root: Path, exts):
    files = sorted(p for p in root.rglob('*') if p.is_file() and p.suffix in exts)
    if MAX_FILES > 0:
        files = files[:MAX_FILES]
    return files

def safe_id(path: Path, root: Path) -> str:
    return str(path.relative_to(root)).replace(os.sep, '__')

def source_family(path: Path, root: Path) -> str:
    parts = path.relative_to(root).parts
    # Family granularity: top 2 dirs where possible (prevents leakage across near-duplicates).
    return '/'.join(parts[:2]) if len(parts) >= 2 else parts[0]

def load_pipelines_from_compare(compare_path: Path):
    fallback_custom = {
        'baseline': '',
        'const_prop': 'sccp',
        'combine_dce': 'instcombine,dce',
        'inline_first': 'cgscc(inline),function(sccp)',
        'sccp_first': 'function(sccp),cgscc(inline)',
    }
    fallback_opt = ['O0', 'O1', 'O2', 'O3', 'Os', 'Oz']

    if not compare_path.exists():
        return fallback_custom, fallback_opt

    tree = ast.parse(compare_path.read_text(encoding='utf-8'))
    custom = None
    opt_levels = None

    for node in tree.body:
        if not isinstance(node, ast.Assign):
            continue
        for target in node.targets:
            if not isinstance(target, ast.Name):
                continue
            if target.id == 'custom_pipelines':
                custom = ast.literal_eval(node.value)
            elif target.id == 'opt_levels':
                opt_levels = ast.literal_eval(node.value)

    merged_opt = sorted(set((opt_levels or []) + fallback_opt))
    return custom or fallback_custom, merged_opt

def get_build_recipe(src: Path):
    recipe = {'cflags': [], 'link_sources': []}
    p = src.as_posix()

    if '/SingleSource/Benchmarks/Polybench/' in p:
        poly_root = next((par for par in src.parents if par.name == 'Polybench'), None)
        if poly_root is not None:
            util_dir = poly_root / 'utilities'
            recipe['cflags'] += ['-I', str(util_dir.resolve()), '-DFP_ABSTOLERANCE=1e-5']
            polybench_c = util_dir / 'polybench.c'
            if polybench_c.exists():
                recipe['link_sources'].append(polybench_c)

    return recipe

def needs_legacy_c_fallback(stderr: str):
    markers = [
        'does not support implicit int',
        '-Wimplicit-int',
        'deprecated-non-prototype',
        'implicit-function-declaration',
    ]
    return any(m in (stderr or '') for m in markers)

def compile_to_base_ir(src: Path, base_ir: Path):
    base_ir.parent.mkdir(parents=True, exist_ok=True)
    recipe = get_build_recipe(src)

    cmd = ['clang', '-O0', '-S', '-emit-llvm', *recipe['cflags'], str(src.resolve()), '-o', str(base_ir.resolve())]
    ok, _, err, elapsed, _ = run_cmd(cmd)
    if ok:
        return True, recipe, '', elapsed

    if needs_legacy_c_fallback(err):
        retry_recipe = {
            'cflags': recipe['cflags'] + LEGACY_C_FLAGS,
            'link_sources': recipe['link_sources'],
        }
        retry = ['clang', '-O0', '-S', '-emit-llvm', *retry_recipe['cflags'], str(src.resolve()), '-o', str(base_ir.resolve())]
        ok2, _, err2, elapsed2, _ = run_cmd(retry)
        if ok2:
            return True, retry_recipe, '', elapsed + elapsed2
        return False, recipe, err2 or err, elapsed + elapsed2

    return False, recipe, err, elapsed

def apply_pipeline(base_ir: Path, opt_ir: Path, exe: Path, recipe: dict, pass_pipeline: str = None, opt_level: str = None):
    t0 = time.perf_counter()
    if pass_pipeline is None and opt_level is None:
        raise ValueError('Either pass_pipeline or opt_level must be provided')

    if pass_pipeline == '':
        shutil.copyfile(base_ir, opt_ir)
        opt_ok, opt_err, opt_to = True, '', False
    elif opt_level is not None:
        opt_ok, _, opt_err, _, opt_to = run_cmd(['opt', f'-{opt_level}', str(base_ir), '-S', '-o', str(opt_ir)])
    else:
        opt_ok, _, opt_err, _, opt_to = run_cmd(['opt', f'-passes={pass_pipeline}', str(base_ir), '-S', '-o', str(opt_ir)])

    if not opt_ok:
        return {'ok': False, 'stage': 'opt', 'reason': opt_err, 'timed_out': opt_to, 'elapsed_s': time.perf_counter() - t0}

    link_cmd = ['clang', str(opt_ir), *recipe.get('cflags', [])]
    link_cmd.extend(str(p.resolve()) for p in recipe.get('link_sources', []))
    link_cmd.extend(['-O0', '-lm', '-o', str(exe)])

    lk_ok, _, lk_err, _, lk_to = run_cmd(link_cmd)
    if not lk_ok:
        return {'ok': False, 'stage': 'link', 'reason': lk_err, 'timed_out': lk_to, 'elapsed_s': time.perf_counter() - t0}

    run_ok = True
    run_reason = ''
    run_elapsed = 0.0
    run_to = False

    if RUN_EXECUTABLE:
        run_ok, _, run_err, run_elapsed, run_to = run_cmd([str(exe.resolve())])
        if not run_ok:
            run_reason = run_err

    size_b = exe.stat().st_size if exe.exists() else None
    return {
        'ok': run_ok,
        'stage': 'run' if not run_ok else 'ok',
        'reason': run_reason,
        'timed_out': run_to,
        'elapsed_s': time.perf_counter() - t0,
        'run_elapsed_s': run_elapsed,
        'exe_size_b': size_b,
    }

def ir_snippet(base_ir: Path, max_lines=120):
    txt = base_ir.read_text(encoding='utf-8', errors='replace').splitlines()
    return '\n'.join(txt[:max_lines])

def make_random_pipeline(pass_pool, kmin, kmax):
    k = RNG.randint(kmin, kmax)
    picks = RNG.sample(pass_pool, k=min(k, len(pass_pool)))
    return ','.join(picks)

def candidate_set(custom_pipelines: dict, opt_levels: list):
    """Build a broad deterministic candidate set before any optional random exploration."""
    cands = []

    # Compare all configured/custom pipelines first.
    for name, p in custom_pipelines.items():
        cands.append({'label': name, 'kind': 'custom', 'pass_pipeline': p, 'opt_level': None})

    # Compare standard LLVM optimization pipelines.
    for lev in sorted(set(opt_levels)):
        cands.append({'label': lev, 'kind': 'opt', 'pass_pipeline': None, 'opt_level': lev})

    # Compare single-pass pipelines from safe pass pool.
    if INCLUDE_SINGLE_PASS_CANDIDATES:
        for p in PASS_POOL:
            cands.append({'label': f'single_{p}', 'kind': 'single', 'pass_pipeline': p, 'opt_level': None})

    # Compare pair-pass pipelines from safe pass pool.
    if INCLUDE_PAIR_PASS_CANDIDATES:
        pairs = [(a, b) for a, b in itertools.permutations(PASS_POOL, 2)]
        if MAX_PAIR_PASS_CANDIDATES > 0:
            pairs = pairs[:MAX_PAIR_PASS_CANDIDATES]
        for a, b in pairs:
            seq = f'{a},{b}'
            cands.append({'label': f'pair_{a}_{b}', 'kind': 'pair', 'pass_pipeline': seq, 'opt_level': None})

    # Optional random exploration.
    if INCLUDE_RANDOM_CANDIDATES and MAX_RANDOM_CANDIDATES > 0:
        seen = set((x['pass_pipeline'], x['opt_level']) for x in cands)
        for i in range(MAX_RANDOM_CANDIDATES):
            rp = make_random_pipeline(PASS_POOL, RANDOM_PIPELINE_MIN_LEN, RANDOM_PIPELINE_MAX_LEN)
            key = (rp, None)
            if key in seen:
                continue
            seen.add(key)
            cands.append({'label': f'rand_{i:02d}', 'kind': 'random', 'pass_pipeline': rp, 'opt_level': None})

    return cands

def objective_key(r):
    # Primary: executable size (lower better), secondary: compile elapsed (lower better).
    return (r.get('exe_size_b', float('inf')), r.get('elapsed_s', float('inf')))

def stable_hash(s: str):
    return hashlib.sha1(s.encode('utf-8')).hexdigest()

print('Utilities loaded.')

Utilities loaded.


In [20]:
# Environment / toolchain sanity checks
tools = ['clang', 'opt']
missing = [t for t in tools if shutil.which(t) is None]
if missing:
    raise RuntimeError(f'Missing required tools: {missing}. Install clang/llvm first.')

source_root = find_source_root()
sources = discover_sources(source_root, SOURCE_EXTS)
custom_pipelines, opt_levels = load_pipelines_from_compare(COMPARE_PY)
all_candidates = candidate_set(custom_pipelines, opt_levels)

print('source_root:', source_root)
print('num_sources:', len(sources))
print('custom pipelines:', list(custom_pipelines.keys()))
print('opt levels:', opt_levels)
print('candidates per file:', len(all_candidates))

source_root: /media/sasank-v/New Volume/Studies/College/VIT/Sem 6/Compiler Design/DA/llvm-test-suite
num_sources: 4452
custom pipelines: ['baseline', 'const_prop', 'combine_dce', 'inline_first', 'sccp_first']
opt levels: ['O0', 'O1', 'O2', 'O3', 'Os', 'Oz']
candidates per file: 300


In [21]:
# Resume support: load already processed source paths
processed = set()
if RESUME and LABELS_CSV.exists():
    prev_rows = read_csv_rows(LABELS_CSV)
    processed = {str(r.get('source', '')) for r in prev_rows if r.get('source')}

print('already processed:', len(processed))
print('remaining:', max(0, len(sources) - len(processed)))

already processed: 0
remaining: 4452


In [ ]:
# Main dataset generation loop
label_rows = []
trial_rows = []

def append_csv_rows(path: Path, rows: list, fieldnames: list):
    if not rows:
        return
    write_header = (not path.exists()) or path.stat().st_size == 0
    with path.open('a', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            w.writeheader()
        w.writerows(rows)

label_fields = [
    'source', 'family', 'status', 'best_label', 'best_kind',
    'best_pass_pipeline', 'best_opt_level', 'best_exe_size_b',
    'best_elapsed_s', 'n_candidates_tested', 'error',
    'ir_snippet', 'source_sha1'
]

trial_fields = [
    'source', 'candidate_label', 'candidate_kind', 'pass_pipeline',
    'opt_level', 'ok', 'stage', 'timed_out', 'exe_size_b',
    'elapsed_s', 'reason'
]

for src in tqdm(sources, desc='files'):
    rel = str(src.relative_to(source_root))
    if RESUME and rel in processed:
        continue

    fam = source_family(src, source_root)
    tid = safe_id(src, source_root)
    tdir = WORK_DIR / tid
    tdir.mkdir(parents=True, exist_ok=True)
    base_ir = tdir / 'base.ll'

    ok_ir, recipe, err_ir, _ = compile_to_base_ir(src, base_ir)
    if not ok_ir:
        label_rows.append({
            'source': rel,
            'family': fam,
            'status': 'compile_base_ir_fail',
            'best_label': '',
            'best_kind': '',
            'best_pass_pipeline': '',
            'best_opt_level': '',
            'best_exe_size_b': '',
            'best_elapsed_s': '',
            'n_candidates_tested': 0,
            'error': (err_ir or '')[:500],
            'ir_snippet': '',
            'source_sha1': stable_hash(rel),
        })

        if len(label_rows) >= 50:
            append_csv_rows(LABELS_CSV, label_rows, label_fields)
            label_rows = []
        continue

    cands = candidate_set(custom_pipelines, opt_levels)
    valid_results = []

    for c in cands:
        label = c['label']
        kind = c['kind']
        pass_pipeline = c['pass_pipeline']
        opt_level = c['opt_level']

        opt_ir = tdir / f'{label}.ll'
        exe = tdir / f'{label}.out'

        r = apply_pipeline(base_ir, opt_ir, exe, recipe, pass_pipeline=pass_pipeline, opt_level=opt_level)

        if WRITE_CANDIDATE_LOG:
            trial_rows.append({
                'source': rel,
                'candidate_label': label,
                'candidate_kind': kind,
                'pass_pipeline': pass_pipeline or '',
                'opt_level': opt_level or '',
                'ok': r['ok'],
                'stage': r.get('stage', ''),
                'timed_out': r.get('timed_out', False),
                'exe_size_b': r.get('exe_size_b', ''),
                'elapsed_s': r.get('elapsed_s', ''),
                'reason': (r.get('reason', '') or '')[:500],
            })

        if r['ok'] and r.get('exe_size_b') is not None:
            valid_results.append({
                'label': label,
                'kind': kind,
                'pass_pipeline': pass_pipeline,
                'opt_level': opt_level,
                **r,
            })

    if not valid_results:
        label_rows.append({
            'source': rel,
            'family': fam,
            'status': 'no_valid_candidate',
            'best_label': '',
            'best_kind': '',
            'best_pass_pipeline': '',
            'best_opt_level': '',
            'best_exe_size_b': '',
            'best_elapsed_s': '',
            'n_candidates_tested': len(cands),
            'error': '',
            'ir_snippet': ir_snippet(base_ir),
            'source_sha1': stable_hash(rel),
        })
    else:
        best = sorted(valid_results, key=objective_key)[0]
        chosen_seq = best['pass_pipeline'] if best['pass_pipeline'] is not None else f"default_{best['opt_level']}"

        label_rows.append({
            'source': rel,
            'family': fam,
            'status': 'ok',
            'best_label': best['label'],
            'best_kind': best['kind'],
            'best_pass_pipeline': chosen_seq,
            'best_opt_level': best['opt_level'] or '',
            'best_exe_size_b': best['exe_size_b'],
            'best_elapsed_s': best['elapsed_s'],
            'n_candidates_tested': len(cands),
            'error': '',
            'ir_snippet': ir_snippet(base_ir),
            'source_sha1': stable_hash(rel),
        })

    # Periodic flush for crash safety.
    if len(label_rows) >= 50:
        append_csv_rows(LABELS_CSV, label_rows, label_fields)
        label_rows = []

    if WRITE_CANDIDATE_LOG and len(trial_rows) >= 500:
        append_csv_rows(CANDIDATES_CSV, trial_rows, trial_fields)
        trial_rows = []

# final flush
append_csv_rows(LABELS_CSV, label_rows, label_fields)
if WRITE_CANDIDATE_LOG:
    append_csv_rows(CANDIDATES_CSV, trial_rows, trial_fields)

print('Labels written to:', LABELS_CSV)
if WRITE_CANDIDATE_LOG:
    print('Candidate log written to:', CANDIDATES_CSV)

files:   0%|          | 0/4452 [00:00<?, ?it/s]

In [ ]:
# Load labels and keep successful rows
labels = read_csv_rows(LABELS_CSV)
if not labels:
    raise RuntimeError('No labels found. Run dataset generation cell first.')

ok = [r for r in labels if r.get('status') == 'ok']
families = {r.get('family', '') for r in ok if r.get('family')}

print('Total rows:', len(labels))
print('Usable rows (status==ok):', len(ok))
print('Families:', len(families))

status_counts = {}
for r in labels:
    st = r.get('status', '')
    status_counts[st] = status_counts.get(st, 0) + 1

print('Status counts:')
for k in sorted(status_counts):
    print(f'  {k:24s} {status_counts[k]}')

In [ ]:
# Build family-aware train/val/test split
families = sorted({r.get('family', '') for r in ok if r.get('family')})
RNG.shuffle(families)

n = len(families)
n_train = int(round(n * TRAIN_RATIO))
n_val = int(round(n * VAL_RATIO))
n_test = n - n_train - n_val
if n_test < 0:
    n_test = 0

train_f = set(families[:n_train])
val_f = set(families[n_train:n_train+n_val])
test_f = set(families[n_train+n_val:])

def split_of_family(f):
    if f in train_f:
        return 'train'
    if f in val_f:
        return 'val'
    return 'test'

for row in ok:
    row['split'] = split_of_family(row.get('family', ''))

split_counts = {'train': 0, 'val': 0, 'test': 0}
for row in ok:
    split_counts[row['split']] += 1

print('Split counts:', split_counts)
print('Example families per split:')
for split_name in ['train', 'val', 'test']:
    fams = sorted({r.get('family', '') for r in ok if r.get('split') == split_name})[:10]
    print(f'  {split_name}: {fams}')

In [ ]:
# Export label JSONL (line-per-file metadata)
with LABELS_JSON.open('w', encoding='utf-8') as f:
    for r in ok:
        obj = {
            'source': r.get('source', ''),
            'family': r.get('family', ''),
            'split': r.get('split', ''),
            'best_pass_pipeline': r.get('best_pass_pipeline', ''),
            'best_exe_size_b': int(float(r.get('best_exe_size_b', 0) or 0)),
            'ir_snippet': r.get('ir_snippet', ''),
            'source_sha1': r.get('source_sha1', ''),
        }
        f.write(json.dumps(obj, ensure_ascii=False) + '\n')

print('Wrote:', LABELS_JSON)

In [ ]:
# Export chat-style finetuning JSONL files with a robust template
ALLOWED_PASSES = [
    'mem2reg', 'instcombine', 'simplifycfg', 'gvn', 'licm',
    'loop-unroll', 'dce', 'inline', 'sccp', 'adce',
    'tailcallelim', 'jump-threading', 'early-cse',
    'sroa', 'reassociate', 'loop-rotate', 'indvars',
]
ALLOWED_PASSES_STR = ', '.join(ALLOWED_PASSES)

SYSTEM_MSG = (
    'You are an LLVM optimization expert for code-size reduction. '
    'Given LLVM IR, output ONLY one pipeline string for opt. '
    'Do not output prose, markdown, code fences, labels, or JSON. '
    'Output must be either: '
    '(A) comma-separated LLVM pass names from the allowed set, or '
    '(B) one token in {default_O0, default_O1, default_O2, default_O3, default_Os, default_Oz}. '
    'No duplicates. Keep natural execution order. Prefer 2-8 passes unless default_O* is best.'
)

def normalize_target(seq: str) -> str:
    s = str(seq or '').strip()
    if s in {'default_O0', 'default_O1', 'default_O2', 'default_O3', 'default_Os', 'default_Oz'}:
        return s
    toks = [t.strip() for t in s.split(',') if t.strip()]
    out = []
    for t in toks:
        if t in ALLOWED_PASSES and t not in out:
            out.append(t)
    return ','.join(out)

def make_record(row):
    src = row.get('source', '')
    fam = row.get('family', '')
    ir = row.get('ir_snippet', '')
    target = normalize_target(row.get('best_pass_pipeline', ''))

    user = (
        f"Task: choose an LLVM optimization pipeline for this program.\n"
        f"Source path: {src}\n"
        f"Program family: {fam}\n"
        "Primary objective: minimize final executable size.\n"
        "Secondary objective: preserve compilability/correctness.\n"
        "Tertiary objective: keep pipeline concise.\n"
        f"Allowed passes: {ALLOWED_PASSES_STR}\n"
        "Output contract:\n"
        "- Output one line only.\n"
        "- Output either a comma-separated pass sequence OR one default_O* token.\n"
        "- Do not include any explanation.\n\n"
        f"LLVM IR snippet:\n{ir}\n"
    )

    return {
        'messages': [
            {'role': 'system', 'content': SYSTEM_MSG},
            {'role': 'user', 'content': user},
            {'role': 'assistant', 'content': target},
        ],
        'meta': {
            'source': src,
            'family': fam,
            'split': row.get('split', ''),
            'best_exe_size_b': int(float(row.get('best_exe_size_b', 0) or 0)),
            'source_sha1': row.get('source_sha1', ''),
            'template_version': 'v2_strict_pipeline_contract',
        }
    }

def write_split(rows, split, out_path):
    sub = [r for r in rows if r.get('split') == split]
    with out_path.open('w', encoding='utf-8') as f:
        for row in sub:
            rec = make_record(row)
            if not rec['messages'][2]['content']:
                continue
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
    return len(sub)

n_train = write_split(ok, 'train', TRAIN_JSONL)
n_val = write_split(ok, 'val', VAL_JSONL)
n_test = write_split(ok, 'test', TEST_JSONL)

print('Wrote:')
print(' ', TRAIN_JSONL, n_train)
print(' ', VAL_JSONL, n_val)
print(' ', TEST_JSONL, n_test)

## Notes

- Start with `MAX_FILES = 100` and `MAX_CANDIDATES_PER_FILE = 12` to validate the pipeline quickly.
- Then scale to full-suite (`MAX_FILES = 0`) for full dataset generation.
- If too many `no_valid_candidate` rows appear, increase timeout and reduce `RUN_EXECUTABLE` strictness.
- You can extend candidate quality by adding mutation/beam search around top pipelines.